# Problema 01: Viga en Voladizo (Euler-Bernoulli)

## Relacion 4 - Sistemas Continuos

**Objetivo:** Calcular las frecuencias naturales y formas modales de una viga en voladizo utilizando la teoria de Euler-Bernoulli.

### Enunciado

Se tiene una viga en voladizo con las siguientes caracteristicas:
- Longitud: $L = 3$ m
- Modulo de elasticidad: $E = 210$ GPa
- Momento de inercia: $I = 8.5 \times 10^{-5}$ m$^4$
- Masa por unidad de longitud: $\mu = 25$ kg/m

Se pide:
1. Resolver la ecuacion de frecuencias para el voladizo
2. Calcular las primeras frecuencias naturales
3. Determinar las formas modales correspondientes

## Datos del Problema

| Parametro | Valor | Unidad |
|-----------|-------|--------|
| $L$ | 3.0 | m |
| $E$ | 210 | GPa |
| $I$ | $8.5 \times 10^{-5}$ | m$^4$ |
| $\mu$ | 25 | kg/m |

In [ ]:
# Importacion de librerias
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# Configuracion de graficos
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

# Datos del problema
L = 3.0       # m - Longitud
E = 210e9     # Pa - Modulo de elasticidad
I = 8.5e-5    # m^4 - Momento de inercia
mu = 25.0     # kg/m - Masa por unidad de longitud

print("DATOS DEL PROBLEMA")
print("="*40)
print(f"Longitud: L = {L} m")
print(f"Modulo de elasticidad: E = {E/1e9} GPa")
print(f"Momento de inercia: I = {I*1e4:.2f} cm^4")
print(f"Masa por unidad de longitud: mu = {mu} kg/m")
print(f"\nRigidez a flexion: EI = {E*I/1e6:.2f} MN.m^2")
print(f"Masa total: m = {mu*L:.1f} kg")

## 1. Teoria de Vigas Euler-Bernoulli

La ecuacion diferencial de vibracion libre para una viga Euler-Bernoulli es:

$$EI \frac{\partial^4 w}{\partial x^4} + \mu \frac{\partial^2 w}{\partial t^2} = 0$$

Asumiendo vibracion armonica $w(x,t) = \phi(x) e^{i\omega t}$:

$$\frac{d^4 \phi}{dx^4} = \beta^4 \phi$$

Donde:
$$\beta^4 = \frac{\mu \omega^2}{EI}$$

La solucion general es:
$$\phi(x) = A\cosh(\beta x) + B\sinh(\beta x) + C\cos(\beta x) + D\sin(\beta x)$$

## 2. Condiciones de Contorno del Voladizo

**En el empotramiento ($x=0$):**
- Desplazamiento nulo: $\phi(0) = 0$
- Giro nulo: $\phi'(0) = 0$

**En el extremo libre ($x=L$):**
- Momento flector nulo: $\phi''(L) = 0$
- Cortante nulo: $\phi'''(L) = 0$

Aplicando estas condiciones se obtiene la **ecuacion de frecuencias**:

$$\cos(\beta L) \cdot \cosh(\beta L) + 1 = 0$$

In [ ]:
def freq_equation(beta_L):
    """
    Ecuacion de frecuencias del voladizo.
    cos(beta*L) * cosh(beta*L) + 1 = 0
    """
    return np.cos(beta_L) * np.cosh(beta_L) + 1

# Graficar la ecuacion de frecuencias para visualizar las raices
beta_L_plot = np.linspace(0.1, 15, 500)
y = freq_equation(beta_L_plot)

plt.figure(figsize=(12, 4))
plt.plot(beta_L_plot, y, 'b-', linewidth=2)
plt.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
plt.xlabel(r'$\beta L$', fontsize=12)
plt.ylabel(r'$\cos(\beta L)\cosh(\beta L) + 1$', fontsize=12)
plt.title('Ecuacion de frecuencias del voladizo', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.ylim(-5, 20)
plt.xlim(0, 15)
plt.show()

print("Las raices son los puntos donde la curva cruza el eje horizontal.")

## 3. Calculo de Frecuencias Naturales

In [ ]:
# Buscar raices de la ecuacion de frecuencias
def find_roots(n_roots=5):
    """
    Encuentra las primeras n raices de la ecuacion de frecuencias.
    """
    beta_L_roots = []
    x_search = np.linspace(0.1, 25, 1000)
    
    for i in range(len(x_search)-1):
        if freq_equation(x_search[i]) * freq_equation(x_search[i+1]) < 0:
            # Hay un cambio de signo -> existe una raiz
            root = brentq(freq_equation, x_search[i], x_search[i+1])
            beta_L_roots.append(root)
            if len(beta_L_roots) >= n_roots:
                break
    
    return np.array(beta_L_roots)

# Encontrar las primeras 5 raices
beta_L_roots = find_roots(5)

print("RAICES DE LA ECUACION DE FRECUENCIAS")
print("="*40)
for n, root in enumerate(beta_L_roots):
    print(f"beta_{n+1}*L = {root:.5f}")

In [ ]:
# Calcular frecuencias naturales
# omega = beta^2 * sqrt(EI/mu)
factor = np.sqrt(E * I / (mu * L**4))

omega = (beta_L_roots**2) * factor  # rad/s
freq_Hz = omega / (2 * np.pi)       # Hz
T = 1 / freq_Hz                      # s

print("FRECUENCIAS NATURALES")
print("="*60)
print(f"Factor sqrt(EI/mu*L^4) = {factor:.3f} rad/s")
print()
print(f"{'Modo':<6} {'beta*L':<12} {'omega (rad/s)':<16} {'f (Hz)':<12} {'T (ms)':<10}")
print("-"*60)
for n in range(len(omega)):
    print(f"{n+1:<6} {beta_L_roots[n]:<12.4f} {omega[n]:<16.2f} {freq_Hz[n]:<12.2f} {T[n]*1000:<10.1f}")

# Verificar relacion entre frecuencias
print("\nRatios de frecuencia respecto al modo fundamental:")
for n in range(1, len(omega)):
    print(f"  omega_{n+1}/omega_1 = {omega[n]/omega[0]:.2f}")

## 4. Formas Modales

La forma modal del voladizo es:

$$\phi_n(x) = \cosh(\beta_n x) - \cos(\beta_n x) - \sigma_n [\sinh(\beta_n x) - \sin(\beta_n x)]$$

Donde:
$$\sigma_n = \frac{\sinh(\beta_n L) - \sin(\beta_n L)}{\cosh(\beta_n L) + \cos(\beta_n L)}$$

In [ ]:
def mode_shape(x, beta, L):
    """
    Calcula la forma modal del voladizo.
    
    Parametros:
        x: coordenada a lo largo de la viga
        beta: parametro de frecuencia (beta_n * L / L)
        L: longitud de la viga
    """
    bL = beta * L
    
    # Coeficiente sigma
    sigma = (np.sinh(bL) - np.sin(bL)) / (np.cosh(bL) + np.cos(bL))
    
    # Forma modal
    phi = (np.cosh(beta*x) - np.cos(beta*x)) - sigma * (np.sinh(beta*x) - np.sin(beta*x))
    
    # Normalizar (maximo = 1)
    phi = phi / np.max(np.abs(phi))
    
    return phi

# Calcular formas modales
x = np.linspace(0, L, 200)
n_modes = min(5, len(beta_L_roots))

print("FORMAS MODALES")
print("="*40)
for n in range(n_modes):
    beta = beta_L_roots[n] / L
    bL = beta * L
    sigma = (np.sinh(bL) - np.sin(bL)) / (np.cosh(bL) + np.cos(bL))
    print(f"Modo {n+1}: sigma_{n+1} = {sigma:.4f}")

## 5. Visualizacion de Resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Ecuacion de frecuencias con raices marcadas
ax1 = axes[0]
beta_L_plot = np.linspace(0.1, 12, 500)
y = freq_equation(beta_L_plot)
ax1.plot(beta_L_plot, y, 'b-', linewidth=2)
ax1.axhline(y=0, color='k', linestyle='--', linewidth=0.5)

# Marcar raices
for i, root in enumerate(beta_L_roots[:3]):
    ax1.plot(root, 0, 'ro', markersize=10)
    ax1.annotate(f'  $\\beta_{i+1}L = {root:.3f}$', (root, 0), fontsize=10)

ax1.set_xlabel(r'$\beta L$', fontsize=12)
ax1.set_ylabel(r'$\cos(\beta L)\cosh(\beta L) + 1$', fontsize=12)
ax1.set_title('Ecuacion de frecuencias', fontweight='bold')
ax1.set_ylim(-5, 15)
ax1.grid(True, alpha=0.3)

# Panel 2: Formas modales
ax2 = axes[1]
colors = ['blue', 'red', 'green', 'orange', 'purple']
for n in range(min(3, n_modes)):
    beta = beta_L_roots[n] / L
    phi = mode_shape(x, beta, L)
    ax2.plot(x, phi, colors[n], linewidth=2, 
             label=f'Modo {n+1} (f = {freq_Hz[n]:.1f} Hz)')

ax2.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
ax2.set_xlabel('Posicion x (m)', fontsize=12)
ax2.set_ylabel('Desplazamiento normalizado', fontsize=12)
ax2.set_title('Formas modales', fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3)

# Panel 3: Espectro de frecuencias
ax3 = axes[2]
modes = np.arange(1, n_modes+1)
bars = ax3.bar(modes, freq_Hz[:n_modes], color='steelblue', 
               edgecolor='black', linewidth=1.2)
ax3.set_xlabel('Numero de modo', fontsize=12)
ax3.set_ylabel('Frecuencia (Hz)', fontsize=12)
ax3.set_title('Espectro de frecuencias', fontweight='bold')
ax3.set_xticks(modes)

# Etiquetas de frecuencia
for i, f in enumerate(freq_Hz[:n_modes]):
    ax3.text(i+1, f + 5, f'{f:.1f}', ha='center', fontsize=10, fontweight='bold')

ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Figura adicional: Visualizacion 3D de los modos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for n in range(3):
    ax = axes[n]
    beta = beta_L_roots[n] / L
    phi = mode_shape(x, beta, L)
    
    # Dibujar viga deformada
    ax.fill_between(x, -0.02, 0.02, color='lightgray', alpha=0.5)
    ax.plot(x, phi * 0.3, colors[n], linewidth=3, label=f'Modo {n+1}')
    
    # Empotramiento
    ax.axvline(x=0, color='black', linewidth=3)
    ax.fill_betweenx([-0.4, 0.4], -0.1, 0, color='gray', alpha=0.5)
    
    ax.set_xlim(-0.2, L + 0.2)
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlabel('Posicion (m)')
    ax.set_title(f'Modo {n+1}: f = {freq_Hz[n]:.1f} Hz', fontweight='bold')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Verificacion con Valores Conocidos

Los valores teoricos de $\beta_n L$ para un voladizo son:

| Modo | $\beta_n L$ (teorico) |
|------|----------------------|
| 1 | 1.8751 |
| 2 | 4.6941 |
| 3 | 7.8548 |
| 4 | 10.9955 |
| 5 | 14.1372 |

In [ ]:
# Valores teoricos
beta_L_teorico = np.array([1.8751, 4.6941, 7.8548, 10.9955, 14.1372])

print("VERIFICACION CON VALORES TEORICOS")
print("="*55)
print(f"{'Modo':<6} {'Calculado':<14} {'Teorico':<14} {'Error (%)':<12}")
print("-"*55)
for n in range(min(len(beta_L_roots), len(beta_L_teorico))):
    error = (beta_L_roots[n] - beta_L_teorico[n]) / beta_L_teorico[n] * 100
    print(f"{n+1:<6} {beta_L_roots[n]:<14.5f} {beta_L_teorico[n]:<14.5f} {error:<12.4f}")

## Interpretacion de Resultados

### Frecuencias naturales
- **Modo 1** (fundamental): $f_1 \approx 20$ Hz. Es el modo de flexion simple con un nodo en el empotramiento.
- Las frecuencias superiores crecen rapidamente: los ratios son aproximadamente 6.27, 17.55, 34.39...
- Esto sigue la relacion $(\beta_n L)^2 / (\beta_1 L)^2$.

### Formas modales
- **Modo 1**: Deformada simple, maximo en el extremo libre. Sin nodos internos.
- **Modo 2**: Un nodo interno (punto de desplazamiento nulo).
- **Modo 3**: Dos nodos internos.
- En general, el modo $n$ tiene $n-1$ nodos internos.

### Aplicaciones practicas
- El modo fundamental domina la respuesta dinamica en la mayoria de casos.
- La frecuencia fundamental es critica para evitar resonancia con excitaciones externas.
- En estructuras reales, el amortiguamiento reduce la amplitud de vibracion en resonancia.